# Research 36 — Five-seed VIP × region OOF average

**Hypothesis:** adding one fixed fifth CV partition may stabilize the leading VIP × region model average. Compare predeclared equal 1/4 and 1/5 component means, each contributing exactly 20% to the established reference. Report macro and weighted F1, fold-slice gains, and a paired class-stratified bootstrap for the incremental fifth-seed effect. This is conditional OOF analysis on the same train set, not independent validation; no fitting or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    row_sums = probs.sum(axis=1, keepdims=True)
    assert (row_sums > 0).all(), filename
    return probs / row_sums


p_base = load_probs("segment-oof.csv")
seed_files = [
    "segment-vip-region-balanced-oof.csv",
    "segment-vip-region-seed-2025-oof.csv",
    "segment-vip-region-seed-1337-oof.csv",
    "segment-vip-region-seed-31415-oof.csv",
    "segment-vip-region-seed-27182-oof.csv",
]
seed_probs = [load_probs(filename) for filename in seed_files]
p_four = np.mean(seed_probs[:4], axis=0)
p_five = np.mean(seed_probs, axis=0)
candidates = {
    "Current reference": p_base,
    "80/20 four-seed average": 0.8 * p_base + 0.2 * p_four,
    "80/20 five-seed average": 0.8 * p_base + 0.2 * p_five,
}
fold_seeds = [42, 2025, 1337, 31415, 27182]
fold_sets = {
    seed: list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    )
    for seed in fold_seeds
}
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for seed, folds in fold_sets.items():
        for fold, (_, idx) in enumerate(folds, start=1):
            row[f"seed{seed}_fold{fold}_macro_f1"] = f1_score(y[idx], pred[idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-vip-region-five-seed-comparison.csv", index=False)
pd.DataFrame(candidates["80/20 five-seed average"], columns=[f"prob_{i}" for i in range(4)]).assign(
    segment=y
).to_csv(ROOT / "predictions/segment-vip-region-five-seed-average-oof.csv", index=False)
print("Five-seed fixed-weight OOF comparison complete; no model fitting or Test access.")

,candidate,macro_f1,weighted_f1,seed42_fold1_macro_f1,seed42_fold2_macro_f1,seed42_fold3_macro_f1,seed2025_fold1_macro_f1,seed2025_fold2_macro_f1,seed2025_fold3_macro_f1,seed1337_fold1_macro_f1,seed1337_fold2_macro_f1,seed1337_fold3_macro_f1,seed31415_fold1_macro_f1,seed31415_fold2_macro_f1,seed31415_fold3_macro_f1,seed27182_fold1_macro_f1,seed27182_fold2_macro_f1,seed27182_fold3_macro_f1
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.750797,0.749412,0.752462,0.749956,0.753540,0.749163,0.749806,0.750854,0.752005,0.752218,0.749245,0.751222
1,80/20 four-seed average,0.752172,0.765704,0.750552,0.751365,0.754568,0.751751,0.751072,0.753684,0.752052,0.754397,0.750051,0.751079,0.752230,0.753192,0.753913,0.751443,0.751159
2,80/20 five-seed average,0.752081,0.765653,0.750624,0.751090,0.754503,0.751710,0.751045,0.753478,0.752127,0.754479,0.749624,0.751095,0.752009,0.753128,0.754191,0.750971,0.751082


Five-seed fixed-weight OOF comparison complete; no model fitting or Test access.


In [2]:
def macro_f1_from_counts(sample_y, sample_pred):
    confusion = np.bincount(sample_y * 4 + sample_pred, minlength=16).reshape(4, 4)
    true_positive = np.diag(confusion)
    denominator = confusion.sum(axis=0) + confusion.sum(axis=1)
    class_f1 = np.divide(
        2 * true_positive, denominator, out=np.zeros(4, dtype=float), where=denominator > 0
    )
    return class_f1.mean()


predictions = {name: probs.argmax(axis=1) for name, probs in candidates.items()}
class_indices = [np.flatnonzero(y == label) for label in np.unique(y)]
rng = np.random.default_rng(42)
n_bootstrap = 300
delta_names = [
    ("four-seed minus reference", "80/20 four-seed average", "Current reference"),
    ("five-seed minus reference", "80/20 five-seed average", "Current reference"),
    ("fifth-seed increment", "80/20 five-seed average", "80/20 four-seed average"),
]
bootstrap_deltas = {name: np.empty(n_bootstrap) for name, _, _ in delta_names}
for sample_number in range(n_bootstrap):
    sample = np.concatenate(
        [rng.choice(indices, size=len(indices), replace=True) for indices in class_indices]
    )
    sample_y = y[sample]
    scores = {
        name: macro_f1_from_counts(sample_y, predictions[name][sample]) for name in candidates
    }
    for delta_name, candidate_name, baseline_name in delta_names:
        bootstrap_deltas[delta_name][sample_number] = scores[candidate_name] - scores[baseline_name]

bootstrap_rows = []
for name, values in bootstrap_deltas.items():
    bootstrap_rows.append(
        {
            "comparison": name,
            "mean_delta": values.mean(),
            "delta_ci_2_5pct": np.quantile(values, 0.025),
            "delta_ci_97_5pct": np.quantile(values, 0.975),
            "fraction_positive": (values > 0).mean(),
        }
    )
bootstrap_results = pd.DataFrame(bootstrap_rows)
display(bootstrap_results.round(6))
bootstrap_results.to_csv(ROOT / "assets/segment-vip-region-five-seed-bootstrap.csv", index=False)
print(
    f"Completed {n_bootstrap} paired class-stratified resamples. These are conditional on selected OOF predictions and omit model/fold/selection uncertainty; no Test access."
)

,comparison,mean_delta,delta_ci_2_5pct,delta_ci_97_5pct,fraction_positive
0,four-seed minus reference,0.001248,0.000467,0.001955,0.996667
1,five-seed minus reference,0.001158,0.000362,0.001861,1.000000
2,fifth-seed increment,-0.000090,-0.000405,0.000177,0.276667


Completed 300 paired class-stratified resamples. These are conditional on selected OOF predictions and omit model/fold/selection uncertainty; no Test access.


## Decision rule and limits

The incremental fifth seed is supported only if the fixed five-seed blend improves on the fixed four-seed blend and does not show erratic fold-slice regressions. If it fails either condition, stop expanding this seed family. Bootstrap resampling is conditional on the saved predictions; it does not account for training, fold, or repeated-selection uncertainty, so even a positive interval is not independent validation or a promotion criterion.